In [1]:
import os
import requests
import logging
import time
from dateutil.relativedelta import relativedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import csv
import pandas as pd 
import numpy as np 

from eutils import EutilsNCBIError, EutilsRequestError
from metapub import PubMedFetcher, pubmedcentral
from datetime import datetime

from Functions import DataRetrieval as DR
from Functions import Metadata as MT
#API_KEY
from Reference_files.keys import API_KEY as API_KEY

In [7]:
# Initialize logger
prefix = "test"+str(datetime.now()).split()[0]
file_handler = logging.FileHandler(f"{prefix}_Examples.log", mode='w')
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)
logging.getLogger().addHandler(file_handler)

# From Query to pubmedCentral full text

In [3]:
# Initialize PubMed fetcher
fetcher = PubMedFetcher()

In [4]:
query_file = "./Reference_files/query.txt"
query = DR.read_query_from_file(query_file)

In [ ]:
start_date = "2002-01-01"
stop_date = None  # Will default to the current date if None\
pmid_array = DR.fetch_pmids_over_period(query_file, start=start_date, stop_date=stop_date, full_text = False)

In [ ]:
## Fetch openaccess PMCs from a query
start_date = "2000-01-01"
stop_date = None  # Will default to the current date if None
pmid_array = DR.fetch_pmids_over_period(query_file, start=start_date, stop_date=stop_date, full_text = True)
pmc_id_list = DR.get_pmcid_for_otherid(pmid_array)
oa_file_list = "Reference_files/oa_file_list.csv"
oa_pmcs = DR.filter_oa_database(oa_file_list, pmc_id_list) # downloads open access database file from NCBI server \size: ~230 mgb\

## Download papers using python requests

In [ ]:
# Create directory
os.makedirs('./Full_text_jsons', exist_ok=True)

for pmcid in oa_pmcids:
    url = f'https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/{pmcid}/unicode'
    response = requests.get(url)
    if response.status_code == 200:
        with open(f'./Full_text_jsons/{pmcid}.json', 'w') as f:
            f.write(response.text)
    else:
        print(f"Failed to download {pmcid}")

###  Or download using powershell \windows/faster\ 

In [ ]:
# Save the oa_pmcids pandas series to a text file that powershell can read(one ID per line)
oa_pmcids.to_csv('oa_pmcids.txt', index=False, header=False)

!powershell -Command "mkdir -p ./Full_text_jsons; Get-Content oa_pmcids.txt | ForEach-Object { Invoke-RestMethod -Uri ('https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/' + $_ + '/unicode') -OutFile ('./Full_text_jsons/' + $_ + '.json') }"

# Or Using bash

In [ ]:
%%bash

mkdir -p ./Full_text_jsons

while IFS= read -r PMCID; do
    url="https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/${PMCID}/unicode"
    curl -s "${url}" > "./Full_text_jsons/${PMCID}.json"
done < full_text_pmc.txt



# Paper Metadata

In [ ]:
paper_meta = DR.fetch_articles_meta(pmid_array)

### add publisher if wanted using Xreff

In [ ]:
added_publishers = DR.process_publishers(paper_meta, email="your_email@rcf.edu")